[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/Trace-Bench/blob/main/notebooks/01_quick_start.ipynb)

# Trace-Bench Smoke Runner (Stub + Real)

This notebook validates Trace-Bench in two modes:

- **StubLLM**: deterministic, no API keys
- **Real LLM**: requires a user-provided API key (via Colab Secrets)

It also shows the standardized run artifacts produced by the CLI.

## Expected Outputs (Quick Verification)

You should see the following signals if the notebook is working correctly:

- **Stub smoke run** completes with a new `runs/<run_id>/` folder.
- `config.snapshot.yaml`, `env.json`, `results.csv`, `events.jsonl` exist in that folder.
- `results.csv` shows at least one row with `task=example:greeting_stub` and `status=ok`.
- **Real-LLM smoke** completes (if API key is set) and `results.csv` shows `status=ok`.
- `pytest -q` ends with `passed` (LLM4AD optimizer tests run only when `OPENAI_API_KEY` is set).

In [ ]:
# Mount Drive (optional) + compute persistent runs_dir
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def bench_dir(project="bench", sub="trace_bench", local="/content/bench"):
    drive = Path("/content/drive/MyDrive")
    root = drive if drive.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

In [ ]:
# Clone repos side-by-side (Trace-Bench + OpenTrace)
!git clone --depth 1 https://github.com/AgentOpt/Trace-Bench.git /content/Trace-Bench 2>/dev/null || (cd /content/Trace-Bench && git pull --ff-only)
!git clone --depth 1 --branch experimental https://github.com/AgentOpt/OpenTrace.git /content/OpenTrace 2>/dev/null || (cd /content/OpenTrace && git pull --ff-only)

%cd /content/Trace-Bench

# System + Python deps
!apt-get update -y && apt-get install -y graphviz
!python -m pip install -U pip
!python -m pip install pyyaml pytest numpy matplotlib graphviz litellm==1.75.0

> **Note:** These notebooks use PYTHONPATH to import from `src/` and typically do not require a runtime restart. If you hit import errors after installs, restart once and rerun the cell.

> **Note:** No runtime restart required. Notebooks use PYTHONPATH to import from src/.

In [ ]:
# Optional: list tasks (external bench discovery)
import sys
sys.path.insert(0, "/content/OpenTrace")
sys.path.insert(0, "/content/Trace-Bench/src")

!PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m trace_bench list-tasks --root benchmarks/LLM4AD/benchmark_tasks | head -n 30

In [ ]:
%%bash
cd /content/Trace-Bench

# Stub smoke (internal example task for deterministic output)
PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m trace_bench run --config configs/smoke.yaml --runs-dir "$RUNS_DIR"

In [ ]:
# Inspect latest run artifacts
import glob, json, pathlib, pandas as pd

latest = sorted(glob.glob(f"{RUNS_DIR}/*"))[-1]
p = pathlib.Path(latest)
print(p)

# Artifacts are under meta/ subdirectory
print((p / "meta" / "config.snapshot.yaml").read_text()[:400])
print(json.loads((p / "meta" / "env.json").read_text()).keys())

pd.read_csv(p / "results.csv").head()

In [ ]:
%%bash
cd /content/Trace-Bench

# Optional: external LLM4AD smoke (may yield low score if template fails)
if [ "${RUN_LLM4AD_SMOKE:-0}" != "1" ]; then
  echo "SKIP: set RUN_LLM4AD_SMOKE=1 to run this optional LLM4AD smoke."
  exit 0
fi

cat > configs/smoke_llm4ad.yaml <<'YAML'
runs_dir: runs
mode: stub
seed: 123
tasks:
  - circle_packing
trainers:
  - PrioritySearch
YAML

PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m trace_bench run --config configs/smoke_llm4ad.yaml --runs-dir "$RUNS_DIR"

## Real LLM (requires API key)

Set one of the following in **Colab Secrets**:
- `OPENROUTER_API_KEY` — routes through OpenRouter (recommended)
- `OPENAI_API_KEY` — direct OpenAI access

Also set `TRACE_LITELLM_MODEL` to your model (e.g. `openrouter/x-ai/grok-4.1-fast`).

In [ ]:
# Load API key from Colab Secrets
import os

try:
    from google.colab import userdata
    # Try OpenRouter first, then OpenAI
    key = userdata.get("OPENROUTER_API_KEY") or ""
    if key:
        os.environ["OPENROUTER_API_KEY"] = key
        os.environ["OPENAI_API_KEY"] = key
        os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
        os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
        print("Using OpenRouter API key")
    else:
        key = userdata.get("OPENAI_API_KEY") or ""
        if key:
            os.environ["OPENAI_API_KEY"] = key
            print("Using OpenAI API key")

    # Also read model from Colab Secrets
    _model = userdata.get("TRACE_LITELLM_MODEL") or ""
    if _model:
        os.environ["TRACE_LITELLM_MODEL"] = _model
except Exception:
    key = os.environ.get("OPENROUTER_API_KEY", "") or os.environ.get("OPENAI_API_KEY", "")

if not key:
    raise RuntimeError("No API key found. Set OPENROUTER_API_KEY or OPENAI_API_KEY in Colab Secrets.")

os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
MODEL = os.environ.get("TRACE_LITELLM_MODEL", "")
if not MODEL:
    raise RuntimeError("Set TRACE_LITELLM_MODEL in Colab Secrets (e.g. openrouter/x-ai/grok-4.1-fast)")

os.environ["TRACE_LITELLM_MODEL"] = MODEL
print(f"Model: {MODEL}")
print("Ready for real-mode runs.")

In [ ]:
%%bash
cd /content/Trace-Bench

# Real-LLM smoke (internal example task)
PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m trace_bench run --config configs/smoke_real.yaml --runs-dir "$RUNS_DIR"

In [ ]:
%%bash
cd /content/Trace-Bench

# Quick smoke tests (skip slow tests and known upstream failures)
if [ "${RUN_SMOKE_TESTS:-0}" != "1" ]; then
  echo "SKIP: set RUN_SMOKE_TESTS=1 to run optional test suite."
  exit 0
fi

PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m pytest tests/m0/ tests/m1/ -q --timeout=120
echo ""
echo "Note: Some test failures may be pre-existing upstream issues, not related to this notebook."